# Symptom onset analysis


In [ ]:
import warnings
warnings.filterwarnings("ignore")

## Data loading


In [ ]:
##DATOS

import pandas as pd

df = pd.read_excel("data_merged-3.xlsx")
#df = df.drop([3, 17, 25, 27]) #quitamos fenotipo 1 y 4
phenotype = df.iloc[:, 167] 
onset_of_symptoms = df.iloc[:, 168] 

#Glucosa
glucose_data = df.iloc[:, 1:167]
subjects = df["Subject"]

print(df.shape)  
print(df.head())  
onset_of_symptoms

In [ ]:
##ESCALAR POR FILAS

from sklearn.preprocessing import StandardScaler
import numpy as np

#Escalamos por columna para manterner la información por zonas de cada sujeto
glucose_scaled = glucose_data.apply(
    lambda row: (row - row.mean()) / row.std(ddof=0), axis=1
)
glucose_scaled = glucose_scaled.fillna(0)

#Añadir sujetos como índices
glucose_scaled.index = df["Subject"]

print(glucose_scaled.shape)
glucose_scaled.head()
df.iloc[:, 1:167] = glucose_scaled

## Descriptive statistics and normality


In [ ]:
import pandas as pd
import numpy as np
from scipy import stats

# Select the column
onset = df.iloc[:, 168].dropna()  # drop NaNs

# Basic descriptive statistics
summary = onset.describe()
print("Descriptive statistics:\n", summary)

# Additional stats
mean = onset.mean()
median = onset.median()
std = onset.std()
min_val = onset.min()
max_val = onset.max()
range_val = max_val - min_val
skewness = onset.skew()
kurtosis = onset.kurtosis()
shapiro_stat, shapiro_p = stats.shapiro(onset)

print(f"\nAdditional statistics:")
print(f"Mean: {mean:.2f}")
print(f"Median: {median:.2f}")
print(f"Standard Deviation: {std:.2f}")
print(f"Min: {min_val:.2f}, Max: {max_val:.2f}, Range: {range_val:.2f}")
print(f"Skewness: {skewness:.3f}")
print(f"Kurtosis: {kurtosis:.3f}")
print(f"Shapiro–Wilk test: Statistic = {shapiro_stat:.4f}, p-value = {shapiro_p:.4f}")


In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import norm, rankdata, shapiro
import matplotlib.pyplot as plt
import seaborn as sns

# --- Nonlinear anamorphosis for a single column ---
def nonlinear_anamorphosis_col(col):
    ranks = rankdata(col)
    uniform = (ranks - 0.5) / len(ranks)
    return norm.ppf(uniform)

# --- Copy your already row-scaled glucose data ---
glucose_scaled_ana = glucose_scaled.copy()

# --- Identify non-normal zones ---
non_normal_zones = []
for zone in glucose_scaled.columns:
    stat, p = shapiro(glucose_scaled[zone])
    if p < 0.05:
        non_normal_zones.append(zone)

print(f"Non-normal zones: {len(non_normal_zones)}")
print("Zones:", non_normal_zones)

# --- Apply nonlinear anamorphosis only to those zones ---
for zone in non_normal_zones:
    glucose_scaled_ana[zone] = nonlinear_anamorphosis_col(glucose_scaled[zone])

# --- Check improvement in skewness ---
print("Original skewness (mean):", glucose_scaled.skew().mean())
print("After selective anamorphosis (mean):", glucose_scaled_ana.skew().mean())

# --- Quick visual check for a few affected zones ---
zones_to_plot = np.random.choice(non_normal_zones, min(3, len(non_normal_zones)), replace=False)
fig, axes = plt.subplots(len(zones_to_plot), 2, figsize=(10, 6))

for i, zone in enumerate(zones_to_plot):
    sns.histplot(glucose_scaled[zone], bins=12, kde=True, ax=axes[i, 0], color='skyblue')
    axes[i, 0].set_title(f"{zone} - Antes")
    sns.histplot(glucose_scaled_ana[zone], bins=12, kde=True, ax=axes[i, 1], color='salmon')
    axes[i, 1].set_title(f"{zone} - Después")

plt.tight_layout()
plt.savefig("anamorphosis.png", dpi=300, bbox_inches="tight") 
plt.show()
df.iloc[:, 1:167] = glucose_scaled_ana

## Correlation analysis


In [ ]:
#spearmanr

import pandas as pd
import scipy.stats as stats

# onset_of_symptoms is a continuous variable (e.g., age)
onset = df.iloc[:, 168] 

results = {}

for col in df.columns[:167]:  # the 166 glucose regions
    x = df[col]
    # Drop NaNs
    valid = (~x.isna()) & (~onset.isna())
    if valid.sum() > 2:  # at least 3 data points
        try:
            rho, p = stats.spearmanr(x[valid], onset[valid])
            results[col] = {"rho": rho, "p_value": p}
        except ValueError:
            continue

# Convert to dataframe
corr_df = pd.DataFrame(results).T
corr_df = corr_df.sort_values("p_value")

# Show top 23 strongest correlations
print(corr_df.head(23))

# Extract top 10 zones
top_zones = corr_df.head(10).index
top_glucose_data = df[top_zones]

# Optional multiple-comparison correction
from statsmodels.stats.multitest import multipletests
corr_df["p_corrected"] = multipletests(corr_df["p_value"], method="fdr_bh")[1]

# Save to CSV
#corr_df.to_csv("onset_glucose_correlations.csv")


In [ ]:
#kendalltau

import pandas as pd
from scipy.stats import kendalltau

# onset_of_symptoms
onset = df.iloc[:, 168]

results = {}

# Loop over all 166 glucose zones
for col in df.columns[:166]:
    x = df[col]
    valid = (~x.isna()) & (~onset.isna())
    if valid.sum() > 2:  # at least 3 data points
        tau, p = kendalltau(x[valid], onset[valid])
        results[col] = {"tau": tau, "p_value": p}

# Convert to DataFrame
kendall_df = pd.DataFrame(results).T
kendall_df = kendall_df.sort_values("p_value")

# Show top 10 ROIs with strongest association
top_kendall_zones = kendall_df.head(10).index
top_kendall_data = df[top_kendall_zones]

print("Top 10 ROIs by Kendall’s tau:")
print(kendall_df.head(15))

# Optional: FDR correction
from statsmodels.stats.multitest import multipletests
kendall_df["p_corrected"] = multipletests(kendall_df["p_value"], method="fdr_bh")[1]


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Your top 10 zones
top_kendall_zones = kendall_df.head(10).index
top_kendall_data = df[top_kendall_zones]

top_zones = top_kendall_zones
top_glucose_data = top_kendall_data
onset = df.iloc[:, 168]

# --- Multi-panel scatter plots ---
n_cols = 2
n_rows = int(np.ceil(len(top_zones) / n_cols))
plt.figure(figsize=(12, n_rows * 4))

for i, zone in enumerate(top_zones):
    plt.subplot(n_rows, n_cols, i + 1)
    sns.regplot(x=top_glucose_data[zone], y=onset, ci=None, scatter_kws={'s':50})
    plt.xlabel(f"Glucose ROI {zone}")
    plt.ylabel("Onset of symptoms")
    plt.title(f"ROI {zone} vs Onset")

plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

top_zones = list(top_zones)  
n_cols = 2
n_rows = int(np.ceil(len(top_zones) / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(12, n_rows * 4))
axes = axes.flatten() 

for i, roi in enumerate(top_zones):
    ax = axes[i]
    sns.scatterplot(x=df[roi], y=df.iloc[:,168], ax=ax)
    sns.regplot(x=df[roi], y=df.iloc[:,168], lowess=True, ci=None, scatter=False, color='red', ax=ax)
    ax.set_xlabel(f"Glucose ROI {roi}")
    ax.set_ylabel("Onset of symptoms")
    ax.set_title(f"ROI {roi} vs Onset (LOWESS)")

# Remove empty subplots if any
for j in range(i+1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()


In [ ]:
#pearsonr

import pandas as pd
import scipy.stats as stats
from statsmodels.stats.multitest import multipletests


onset = df.iloc[:, 168] 

results = {}

for col in df.columns[:167]:  # the 166 glucose regions
    df[col] = pd.to_numeric(df[col], errors='coerce')
    x = df[col]
    # Drop NaNs
    valid = (~x.isna()) & (~onset.isna())
    if valid.sum() > 2:  # at least 3 data points
        try:
            r, p = stats.pearsonr(x[valid], onset[valid])
            results[col] = {"r": r, "p_value": p}
        except ValueError:
            continue

# Convert to dataframe
corr_df = pd.DataFrame(results).T
corr_df = corr_df.sort_values("p_value")

# Show top strongest correlations
print(corr_df.head(60))

In [ ]:
import pandas as pd

# Read the file (assuming tab-separated or space-separated)
zones_df = pd.read_csv("zones_numbered.txt", sep=None, engine="python", header=None)
zones_df.columns = ["ZoneNumber", "Tag"]

# Convert ZoneNumber to string if your df columns are strings
zones_df["ZoneNumber"] = zones_df["ZoneNumber"].astype(str)

# Create mapping dictionary
anatomical_mapping = dict(zip(zones_df["ZoneNumber"], zones_df["Tag"]))

# Example: print first 5 entries
print(list(anatomical_mapping.items())[:15])

df_renamed = df.rename(columns=anatomical_mapping)

In [ ]:
import pandas as pd
import scipy.stats as stats
from statsmodels.stats.multitest import multipletests
import matplotlib.pyplot as plt
import numpy as np

# --- Configuración ---
onset = df.iloc[:, 168]  # Columna "edad de inicio de síntomas"
n_zones = 166            # Número de ROIs cerebrales de glucosa

# --- Cálculo de correlación de Pearson para todas las regiones ---
resultados = {}
for col in df.columns[:167]:
    x = df[col]
    valid = (~x.isna()) & (~onset.isna())
    if valid.sum() > 2:
        try:
            r, p = stats.pearsonr(x[valid], onset[valid])
            resultados[col] = {"r": r, "p_value": p}
        except Exception:
            continue

# --- Construcción de la tabla de resultados ---
corr_df = pd.DataFrame(resultados).T
corr_df = corr_df.sort_values("p_value")

# --- Corrección FDR (Benjamini-Hochberg) ---
corr_df["p_corregido"] = multipletests(corr_df["p_value"], method="fdr_bh")[1]

# --- Identificar ROIs significativas (FDR < 0.05) ---
significant = corr_df[corr_df["p_corregido"] < 0.05]
print(f"ROIs significativas tras FDR: {len(significant)} / {n_zones}")

# --- Crear etiquetas con número + nombre anatómico ---
def make_label(zone_number):
    tag = anatomical_mapping.get(str(zone_number), "Desconocida")
    return f"{zone_number} - {tag}"

# --- Visualización: principales correlaciones ---
top_n = 22  # Número de ROIs a mostrar
top_corr = corr_df.head(top_n)
top_corr_rev = top_corr.iloc[::-1]  # Revertir para el gráfico horizontal

# Crear etiquetas para eje Y
y_labels = [make_label(zone) for zone in top_corr_rev.index]

# Colores según signo de correlación
colors = np.where(top_corr_rev["r"] > 0, "#4682B4", "#CD5C5C")  # Azul y rojo suave

# --- Crear el gráfico ---
plt.figure(figsize=(12, 8))
bars = plt.barh(y_labels, top_corr_rev["r"], color=colors, edgecolor='black', height=0.7)
plt.axvline(0, color='k', linestyle='--', linewidth=1)
plt.title(f"Correlaciones entre glucosa cerebral y edad de inicio de síntomas", fontsize=16, pad=15)
plt.xlabel("Coeficiente de correlación de Pearson (r)", fontsize=12)
plt.ylabel("Región cerebral", fontsize=12)
plt.xticks(fontsize=10)
plt.yticks(fontsize=10)
plt.tight_layout()
plt.grid(axis='x', linestyle=':', alpha=0.7)
plt.savefig("SintomasPearson.png", dpi=300, bbox_inches='tight')
plt.show()


## Binary classification task

## Early versus late symptom onset


In [ ]:
import numpy as np

onset = df.iloc[:, 168]  # edad de inicio de síntomas
median_onset = np.median(onset)

# Clases binarias invertidas: 0 = mayor edad, 1 = menor edad
y_bin = (onset <= median_onset).astype(int)
print(f"Distribución de clases:\n{np.bincount(y_bin)}")


In [ ]:
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import numpy as np

# --- Data ---

top_rois = top_corr.index.tolist()
X = df[top_rois] 

# --- PCA ---
pca = PCA()
X_pca = pca.fit_transform(X)

# Explained variance
explained_variance = pca.explained_variance_ratio_
cumulative_variance = np.cumsum(explained_variance)

# Print results
for i, (var, cum_var) in enumerate(zip(explained_variance, cumulative_variance), start=1):
    print(f"PC{i}: {var:.3f} (cumulative: {cum_var:.3f})")

# --- Optional: plot explained variance ---
plt.figure(figsize=(8,5))
plt.bar(range(1, len(explained_variance)+1), explained_variance, alpha=0.6, label='Individual')
plt.step(range(1, len(cumulative_variance)+1), cumulative_variance, where='mid', label='Acumulada')
plt.xlabel('Componente principal')
plt.ylabel('Porcentaje de la variabilidad explicada')
plt.title('Variabilidad explicada por PCA')
plt.legend()
plt.savefig("PCA.png", dpi=300, bbox_inches="tight") 
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score
import pandas as pd

onset = df.iloc[:, 168]  # age of onset
median_onset = np.median(onset)
y_bin = (onset > median_onset).astype(int)
print(f"Class distribution:\n{np.bincount(y_bin)}")

# --- Define ranked zones ---
top_zones_all = top_corr.index.tolist()  # your top correlated ROIs

# --- Parameters ---
top_range = range(2, 23)   # test from 2 to 22 top zones
n_clusters = len(np.unique(y_bin))  # should be 2 for binary
ari_list = []

# --- Loop over different top zone counts ---
for top_n in top_range:
    top_zones = top_zones_all[:top_n]
    X_top = df[top_zones]

    # --- PCA dimensionality reduction ---
    X_pca = PCA(n_components=2, random_state=42).fit_transform(X_top)

    # --- KMeans clustering ---
    clusters = KMeans(n_clusters=n_clusters, random_state=42).fit_predict(X_pca)

    # --- Adjusted Rand Index vs true labels ---
    ari = adjusted_rand_score(y_bin, clusters)
    ari_list.append(ari)

# --- Plot results ---
plt.figure(figsize=(8, 5))
plt.plot(top_range, ari_list, marker='o', color='purple', label='Top correlated zones')
plt.xlabel('Number of top zones used')
plt.ylabel('Adjusted Rand Index (ARI)')
plt.title('Clustering quality vs number of top correlated zones')
plt.xticks(top_range)
plt.ylim(-0.1, 1.0)
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

# --- Optional: print the best result ---
best_n = top_range[np.argmax(ari_list)]
print(f"Best clustering ARI = {max(ari_list):.3f} using top {best_n} zones.")


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score

# --- Etiquetas binarias según edad de inicio de síntomas ---
onset = df.iloc[:, 168]
median_onset = np.median(onset)
y_bin = (onset > median_onset).astype(int)
print(f"Distribución de clases: {np.bincount(y_bin)}")

# --- Selección de las zonas más correlacionadas ---
top_n = 22
top_zones = top_corr.head(top_n).index.tolist()
X_top = df[top_zones]
print(f"Usando las {top_n} zonas más correlacionadas: {top_zones}")

# --- PCA a 2 componentes ---
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_top)

# --- Clustering KMeans ---
n_clusters = len(np.unique(y_bin))
kmeans = KMeans(n_clusters=n_clusters, random_state=42)
clusters = kmeans.fit_predict(X_pca)

# --- Evaluación del clustering ---
ari = adjusted_rand_score(y_bin, clusters)
print(f"Índice de Rand ajustado (ARI): {ari:.3f}")



# --- Gráfico de dispersión con proyección PCA ---
plt.figure(figsize=(9,6))
scatter = plt.scatter(
    X_pca[:,0], X_pca[:,1], 
    c=clusters, cmap='tab10', s=80, edgecolor='k', alpha=0.8
)

# Añadir anotaciones con etiqueta de grupo de edad (0=menor, 1=mayor)
for i, label in enumerate(y_bin):
    plt.annotate(str(int(label)), (X_pca[i,0]+0.02, X_pca[i,1]+0.02), fontsize=8)

# Etiquetas y título en español
plt.xlabel("PC1", fontsize=12)
plt.ylabel("PC2", fontsize=12)
plt.title(f"KMeans usando las {top_n} zonas más correlacionadas con PCA 2\nARI = {ari:.3f}", fontsize=14, pad=15)

# Leyenda formal de clusters
handles, labels = scatter.legend_elements()
plt.legend(handles, [f"Cluster {i}" for i in range(n_clusters)], title="Clusters", fontsize=10)

plt.grid(linestyle=':', alpha=0.6)
plt.tight_layout()
plt.savefig("SintomasCluster.png", dpi=300, bbox_inches='tight')

plt.show()

# --- Tabla de contingencia: clusters vs grupo de edad ---
ct = pd.crosstab(y_bin, clusters)
print("\nTabla de contingencia (grupo real vs clusters predichos):")
print(ct)


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import kruskal

# --- Preparar DataFrame ---
df_plot = df.copy()
df_plot["cluster"] = clusters
df_plot["onset"] = onset
median_onset = np.median(onset)
df_plot["grupo_real"] = (onset < median_onset).astype(int)  # 0 = menor, 1 = mayor

# --- Kruskal-Wallis por cluster ---
groups_cluster = [group["onset"].values for _, group in df_plot.groupby("cluster")]
H_cluster, p_cluster = kruskal(*groups_cluster)
n_cluster = len(df_plot)
k_cluster = df_plot["cluster"].nunique()
eta_sq_cluster = (H_cluster - k_cluster + 1) / (n_cluster - k_cluster)

# --- Kruskal-Wallis por grupo real ---
groups_real = [group["onset"].values for _, group in df_plot.groupby("grupo_real")]
H_real, p_real = kruskal(*groups_real)
n_real = len(df_plot)
k_real = df_plot["grupo_real"].nunique()
eta_sq_real = (H_real - k_real + 1) / (n_real - k_real)

# --- Configurar figura con dos subplots ---
plt.figure(figsize=(12,6))

# Boxplot para grupo real
plt.subplot(1,2,1)
sns.boxplot(x="grupo_real", y="onset", data=df_plot, palette="Set2")
sns.swarmplot(x="grupo_real", y="onset", data=df_plot, color="k", alpha=0.6)
plt.title(f"Grupo real (edad de inicio de síntomas)\nH={H_real:.2f}, p={p_real:.2e}, η²={eta_sq_real:.2f}", fontsize=12)
plt.xlabel("Grupo de edad (0=mayor, 1=menor)")
plt.ylabel("Edad de inicio de síntomas")

# Boxplot para clusters
plt.subplot(1,2,2)
sns.boxplot(x="cluster", y="onset", data=df_plot, palette="Set3")
sns.swarmplot(x="cluster", y="onset", data=df_plot, color="k", alpha=0.6)
plt.title(f"Clusters KMeans\nH={H_cluster:.2f}, p={p_cluster:.2e}, η²={eta_sq_cluster:.2f}", fontsize=12)
plt.xlabel("Cluster")
plt.ylabel("Edad de inicio de síntomas")

plt.tight_layout()
plt.savefig("SintomasBoxplot.png", dpi=300, bbox_inches='tight')

plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# --- Loadings de PCA1 ---
loadings = pd.Series(pca.components_[0], index=X_top.columns)  # PC1

# Ordenar por magnitud absoluta (de mayor a menor)
loadings_sorted = loadings.reindex(loadings.abs().sort_values(ascending=False).index)

# Crear etiquetas combinadas
def make_label(zone_number):
    tag = anatomical_mapping.get(str(zone_number), "Desconocida")
    return f"{zone_number} - {tag}"

y_labels = [make_label(zone) for zone in loadings_sorted.index]

# Colores según signo real
colors = np.where(loadings_sorted > 0, "#4682B4", "#CD5C5C")  # Azul positivo, Rojo negativo

# Invertir los valores y los colores para que la barra más grande quede arriba
loadings_plot = loadings_sorted[::-1]
y_labels_plot = y_labels[::-1]
colors_plot = colors[::-1]

# --- Gráfico horizontal ---
plt.figure(figsize=(12,8))
plt.barh(y_labels_plot, loadings_plot, color=colors_plot, edgecolor='black', height=0.7)
plt.axvline(0, color='k', linestyle='--', linewidth=1)
plt.title("Loadings de PCA1 para las zonas cerebrales más correlacionadas", fontsize=16, pad=15)
plt.xlabel("Loading PCA1", fontsize=12)
plt.ylabel("Zona cerebral", fontsize=12)
plt.xticks(fontsize=10)
plt.yticks(fontsize=10)
plt.grid(axis='x', linestyle=':', alpha=0.7)
plt.tight_layout()
plt.savefig("PCA1_Loadings_TFM.png", dpi=300, bbox_inches='tight')
plt.show()
